In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

e:\Chatbot_Rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# read all the pdf inside directory

def process_all_pdf(pdf_directory):
    """process the all the pdf files in directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"found {len(pdf_files)} pdf files to process")
    
    for pdf_file in pdf_files:
        print(f"\nprocessing:{pdf_file.name}")
        try:
            loader = PyPDFLoader(pdf_file)
            document = loader.load()
            
            for doc in document:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(document)
            print(f"loaded {len(document)} pages")
        except Exception as e:
            print(f"error:{e}")
            
    print(f"\n total documents loaded:{len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdf('../Data/Chatbot_pdfs')

found 4 pdf files to process

processing:2604.10637v1.pdf
loaded 11 pages

processing:2604.11195v1.pdf
loaded 15 pages

processing:2604.11234v1.pdf
loaded 17 pages

processing:2604.11402v1.pdf
loaded 19 pages

 total documents loaded:62


In [3]:
# convert this documenst into chunks
def split_documents(document,chunk_size=1000,chunk_overlap=200):
    "split documents into chunks for better Rag performence"
    TextSplitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=['\n\n','\n'," ",""]
    )
    
    split_docs = TextSplitter.split_documents(document)
    print(f"split {len(document)} documenst into {len(split_docs)}")
    
    if split_docs:
        print("\n example of chunk")
        print(f"content: {split_docs[0].page_content[:200]}")
        print(f"metadata: {split_docs[0].metadata}")
        
    return split_docs

In [4]:
chunks = split_documents(all_pdf_documents)
chunks

split 62 documenst into 367

 example of chunk
content: 1
Language Prompt vs. Image Enhancement: Boosting
Object Detection With CLIP in Hazy Environments
Jian Pang, Bingfeng Zhang, Jin Wang, Baodi Liu, Dapeng Tao, Weifeng Liu,
Abstract—Object detection in 
metadata: {'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Jian Pang; Bingfeng Zhang; Jin Wang; Baodi Liu; Dapeng Tao; Weifeng Liu', 'doi': 'https://doi.org/10.48550/arXiv.2604.10637', 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Language Prompt vs. Image Enhancement: Boosting Object Detection With CLIP in Hazy Environments', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2604.10637v1', 'source': '..\\Data\\Chatbot_pdfs\\2604.10637v1.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': '2604.10637v1.pdf', 'file_typ

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Jian Pang; Bingfeng Zhang; Jin Wang; Baodi Liu; Dapeng Tao; Weifeng Liu', 'doi': 'https://doi.org/10.48550/arXiv.2604.10637', 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Language Prompt vs. Image Enhancement: Boosting Object Detection With CLIP in Hazy Environments', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2604.10637v1', 'source': '..\\Data\\Chatbot_pdfs\\2604.10637v1.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': '2604.10637v1.pdf', 'file_type': 'pdf'}, page_content='1\nLanguage Prompt vs. Image Enhancement: Boosting\nObject Detection With CLIP in Hazy Environments\nJian Pang, Bingfeng Zhang, Jin Wang, Baodi Liu, Dapeng Tao, Weifeng Liu,\nAbstract—Object detection in hazy environments 

### converting splitted documents into embedding and store in vectordb

In [6]:

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import Any,List,Tuple,Dict
from sklearn.metrics.pairwise import cosine_similarity


In [8]:
class EmbeddingManager:
    "handle the document embedding generation using sentence transformer"
    
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """
        Initalize the embedding manager
        
        Args:
        model_name : hugging face model name for sentence embeddings
        
        """
        
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        "load sentence transformer model"
        try:
            print(f"loading embidding model:{self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print("model loaded successfully")
        except Exception as e:
            print(f"failed load model {self.model_name}: {e}")
            
            
    def generate_embeddings(self,texts: List[str]) -> np.ndarray:
        """
        Generate embeddings from list of texts
        
        Args:
        texts: list of texts
        
        returns: numpy array of embeddings
        
        """
        
        if not self.model:
            raise ValueError("model not loaded")
        print(f"Generating embeddings for {len(texts)} texts ..")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings shape:{embeddings.shape}")
        return embeddings
    
embedding_managaer = EmbeddingManager()
embedding_managaer
            
        

loading embidding model:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4387.75it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model loaded successfully


In [ ]:

class VectorDB:
    "handles document of embeddings  in a chromadb vectorstore"
    
    def __init__(self,collection_name:str="datasource",persist_directory:str='../Data/vector_store'):
        """
        initialize vector store
        
        Args:
        collection name : Name of the chromadb collection
        persist_directory:Directory to persist vector store
        
        """
        self.collection_name=collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        "Initialize chromadb client and collection"
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            
            #Get or create collection
            
            self.collection = self.client.get_or_create_collection(
                name =self.collection_name,
                metadata={"description":"pdf document embedding for RAG"}
            )
            
            print(f"vector store initialized collection:{self.collection_name}")
            print(f"existing file present in a collection:{self.collection.count()}")
        except Exception as e:
            print(f"Error in a initializing vector store")
            raise
        
    
    def add_documents(self, documents: List[Any],embeddings:np.ndarray):
        """
         adding the documents and embedding to the vector store
         
         Args:
         documents: list of langchain documents
         embedding : corresponding embedding for documents 
        """
        
        if len(documents) != len(embeddings):
            raise ValueError("numer of documents must match with number of documents")
        
        print(f"adding {len(documents)} to the vector store")
        
        # prepare data for chromadb
        ids = []
        embedding_list = []
        metadatas = []
        document_text = []
        
        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            # Generating unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            #meatadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] =i
            metadata['context_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            #document content
            document_text.append(doc.page_content)
            
            #Embeddings
            embedding_list.append(embedding.tolist())
            
            
        # Add to collection 
        
        try:
            self.collection.add(
                ids = ids,
                embeddings=embedding_list,
                metadatas = metadatas,
                documents = document_text
            )
            
            print(f"succesfully added {len(documents)} documents to vector store")
            print(f"total documents added to collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"failed to add collection in vector store:{e}")
            raise
        
vectorstore = VectorDB()
vectorstore
            
            
            
    
    

vector store initialized collection:datasource
existing file present in a collection:0


In [12]:
# convert texts into embeddings
texts = [doc.page_content for doc in chunks]

#generating embeddings
embeddings = embedding_managaer.generate_embeddings(texts)

# store the embeddings in vector store

vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 367 texts ..


Batches: 100%|██████████| 12/12 [00:11<00:00,  1.06it/s]

Generated embeddings shape:(367, 384)
adding 367 to the vector store
failed to add collection in vector store:Collection.add() got an unexpected keyword argument 'metadata'


TypeError: Collection.add() got an unexpected keyword argument 'metadata'